In [1]:
"""
MBDD2025 — Script 0: Pre-flight Check
Run this BEFORE anything else. Checks all dependencies,
paths, GPU, dataset integrity, and disk space.
"""

import sys
import os
from pathlib import Path

PASS = "  [OK]"
FAIL = "  [FAIL]"
WARN = "  [WARN]"

errors   = []
warnings = []

print("=" * 55)
print("  MBDD2025 PRE-FLIGHT CHECK")
print("=" * 55)

# ─── 1. PYTHON VERSION ────────────────────────────────────────
print("\n[1] Python Version")
major, minor = sys.version_info[:2]
if major == 3 and minor >= 8:
    print(f"{PASS} Python {major}.{minor} (compatible)")
else:
    print(f"{FAIL} Python {major}.{minor} — need 3.8 or higher")
    errors.append("Python version too old")

# ─── 2. PACKAGES ─────────────────────────────────────────────
print("\n[2] Required Packages")
packages = {
    "torch":        "torch",
    "torchvision":  "torchvision",
    "ultralytics":  "ultralytics",
    "cv2":          "opencv-python",
    "yaml":         "pyyaml",
    "pandas":       "pandas",
    "matplotlib":   "matplotlib",
    "numpy":        "numpy",
}
missing_pkgs = []
for import_name, pkg_name in packages.items():
    try:
        __import__(import_name)
        print(f"{PASS} {pkg_name}")
    except ImportError:
        print(f"{FAIL} {pkg_name} — run: pip install {pkg_name}")
        missing_pkgs.append(pkg_name)

if missing_pkgs:
    errors.append(f"Missing packages: {', '.join(missing_pkgs)}")

# ─── 3. GPU / CUDA ────────────────────────────────────────────
print("\n[3] GPU & CUDA")
try:
    import torch
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"{PASS} GPU   : {gpu_name}")
        print(f"{PASS} VRAM  : {vram_gb:.1f} GB")
        if vram_gb < 4:
            warnings.append("Less than 4GB VRAM — reduce BATCH to 4 in script 3")
            print(f"{WARN} VRAM low — set BATCH=4 in script3_train.py")
        print(f"{PASS} CUDA  : {torch.version.cuda}")
    else:
        print(f"{FAIL} No CUDA GPU detected — training will be extremely slow on CPU")
        errors.append("No GPU/CUDA detected")
except Exception as e:
    print(f"{FAIL} Could not check GPU: {e}")
    errors.append("GPU check failed")

# ─── 4. FILE PATHS ───────────────────────────────────────────
print("\n[4] File Paths")

BASE_DIR    = Path(r"C:\Users\DELL\Desktop\aditya project\MBDD2025")
OUTPUT_DIR  = Path(r"C:\Users\DELL\Desktop\aditya project\MBDD2025_split")
YAML_PATH   = Path(r"C:\Users\DELL\Desktop\aditya project\mbdd2025.yaml")
WEIGHTS_DIR = Path(r"C:\Users\DELL\Desktop\aditya project\runs\yolov8m_mbdd2025\weights")
RUNS_DIR    = Path(r"C:\Users\DELL\Desktop\aditya project\runs")

# Dataset base
if BASE_DIR.exists():
    print(f"{PASS} Dataset folder exists : {BASE_DIR}")
else:
    print(f"{FAIL} Dataset folder missing : {BASE_DIR}")
    errors.append(f"Dataset folder not found: {BASE_DIR}")

# Images folder
img_dir = BASE_DIR / "JPEGImages"
if img_dir.exists():
    imgs = list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.jpeg")) + list(img_dir.glob("*.png"))
    print(f"{PASS} JPEGImages/ folder exists : {len(imgs)} images found")
    if len(imgs) == 0:
        print(f"{WARN} JPEGImages/ folder is empty!")
        warnings.append("JPEGImages/ folder is empty")
else:
    print(f"{FAIL} JPEGImages/ folder missing : {img_dir}")
    errors.append("JPEGImages/ folder not found")

# Labels folder
lbl_dir = BASE_DIR / "Labels"
if lbl_dir.exists():
    lbls = list(lbl_dir.glob("*.txt"))
    print(f"{PASS} Labels/ folder exists     : {len(lbls)} label files found")
    if len(lbls) == 0:
        print(f"{WARN} Labels/ folder is empty!")
        warnings.append("Labels/ folder is empty")
else:
    print(f"{FAIL} Labels/ folder missing : {lbl_dir}")
    errors.append("Labels/ folder not found")

# Check image-label pairing
try:
    img_stems = {f.stem for f in imgs}
    lbl_stems = {f.stem for f in lbls}
    matched   = img_stems & lbl_stems
    unmatched = img_stems - lbl_stems
    print(f"{PASS} Matched pairs          : {len(matched)}")
    if unmatched:
        print(f"{WARN} Images without labels  : {len(unmatched)} (will be skipped)")
        warnings.append(f"{len(unmatched)} images have no matching label")
except:
    pass

# Split output (only warn if missing, it gets created by script 1)
if OUTPUT_DIR.exists():
    print(f"{PASS} Split output folder exists (script 1 already run)")
else:
    print(f"{WARN} Split folder not yet created (normal — run script 1 first)")

# YAML
if YAML_PATH.exists():
    print(f"{PASS} mbdd2025.yaml exists")
else:
    print(f"{WARN} mbdd2025.yaml not yet created (normal — run script 2 first)")

# Weights (only relevant after training)
best_pt = WEIGHTS_DIR / "best.pt"
if best_pt.exists():
    print(f"{PASS} best.pt found (training already done)")
else:
    print(f"{WARN} best.pt not found yet (normal — run script 3 first)")

# ─── 5. DISK SPACE ───────────────────────────────────────────
print("\n[5] Disk Space")
try:
    import shutil
    drive = Path(r"C:\Users\DELL\Desktop")
    total, used, free = shutil.disk_usage(drive)
    free_gb = free / 1e9
    print(f"      Free space on C: drive : {free_gb:.1f} GB")
    if free_gb < 5:
        print(f"{FAIL} Less than 5GB free — training may crash!")
        errors.append("Insufficient disk space (< 5GB free)")
    elif free_gb < 10:
        print(f"{WARN} Less than 10GB free — monitor disk during training")
        warnings.append("Low disk space (< 10GB free)")
    else:
        print(f"{PASS} Sufficient disk space")
except Exception as e:
    print(f"{WARN} Could not check disk space: {e}")

# ─── 6. LABEL FORMAT SANITY CHECK ────────────────────────────
print("\n[6] Label Format Sanity Check (first 3 files)")
try:
    sample_lbls = list(lbl_dir.glob("*.txt"))[:3]
    label_ok = True
    for lf in sample_lbls:
        with open(lf) as f:
            lines = f.readlines()
        for line in lines[:2]:
            parts = line.strip().split()
            if len(parts) != 5:
                print(f"{FAIL} Bad format in {lf.name}: '{line.strip()}' (expected 5 values)")
                errors.append(f"Bad label format in {lf.name}")
                label_ok = False
                break
            cls_id = int(parts[0])
            coords = [float(x) for x in parts[1:]]
            if not all(0.0 <= c <= 1.0 for c in coords):
                print(f"{FAIL} Coordinates out of range in {lf.name} (should be 0–1)")
                errors.append("Label coordinates not normalized")
                label_ok = False
                break
            if cls_id not in range(5):
                print(f"{WARN} Class ID {cls_id} in {lf.name} — expected 0–4")
                warnings.append(f"Unexpected class ID {cls_id}")
        if label_ok:
            print(f"{PASS} {lf.name}")
except Exception as e:
    print(f"{WARN} Could not check labels: {e}")

# ─── SUMMARY ─────────────────────────────────────────────────
print("\n" + "=" * 55)
print("  SUMMARY")
print("=" * 55)

if errors:
    print(f"\n  ✗ {len(errors)} ERROR(S) — fix before running:")
    for e in errors:
        print(f"    • {e}")
else:
    print(f"\n  ✓ No errors found!")

if warnings:
    print(f"\n  ⚠  {len(warnings)} WARNING(S):")
    for w in warnings:
        print(f"    • {w}")

if not errors:
    print("\n  ► You are ready to run scripts 1 → 5 in order.")
else:
    print("\n  ► Fix the errors above before proceeding.")

print("=" * 55)

  MBDD2025 PRE-FLIGHT CHECK

[1] Python Version
  [OK] Python 3.13 (compatible)

[2] Required Packages
  [FAIL] torch — run: pip install torch
  [FAIL] torchvision — run: pip install torchvision
  [FAIL] ultralytics — run: pip install ultralytics
  [FAIL] opencv-python — run: pip install opencv-python
  [OK] pyyaml
  [OK] pandas
  [OK] matplotlib
  [OK] numpy

[3] GPU & CUDA
  [FAIL] Could not check GPU: No module named 'torch'

[4] File Paths
  [FAIL] Dataset folder missing : C:\Users\DELL\Desktop\aditya project\MBDD2025
  [FAIL] JPEGImages/ folder missing : C:\Users\DELL\Desktop\aditya project\MBDD2025/JPEGImages
  [FAIL] Labels/ folder missing : C:\Users\DELL\Desktop\aditya project\MBDD2025/Labels
  [WARN] Split folder not yet created (normal — run script 1 first)
  [WARN] mbdd2025.yaml not yet created (normal — run script 2 first)
  [WARN] best.pt not found yet (normal — run script 3 first)

[5] Disk Space
  [WARN] Could not check disk space: [Errno 2] No such file or directory: 'C

In [2]:
"""
MBDD2025 — Script 1: Split Dataset
Splits local images/ and labels/ into train/val/test folders.
Run this FIRST in Jupyter Notebook.

Dataset path: C:\\Users\\DELL\\Desktop\\aditya project\\MBDD2025
"""

import os
import shutil
import random
from pathlib import Path

# ─── CONFIG ───────────────────────────────────────────────────────────────────
BASE_DIR    = r"C:\Users\DELL\Desktop\aditya project\MBDD2025"
OUTPUT_DIR  = r"C:\Users\DELL\Desktop\aditya project\MBDD2025_split"
TRAIN_RATIO = 0.80
VAL_RATIO   = 0.10
TEST_RATIO  = 0.10
SEED        = 42
# ──────────────────────────────────────────────────────────────────────────────

IMG_DIR = Path(BASE_DIR) / "JPEGImages"
LBL_DIR = Path(BASE_DIR) / "Labels"

# ── Step 1: Verify folders exist ──────────────────────────────────────────────
print(f"[INFO] Looking for images in : {IMG_DIR}")
print(f"[INFO] Looking for labels in : {LBL_DIR}")

if not IMG_DIR.exists():
    raise FileNotFoundError(f"[ERROR] images/ folder not found at: {IMG_DIR}")
if not LBL_DIR.exists():
    raise FileNotFoundError(f"[ERROR] labels/ folder not found at: {LBL_DIR}")

# ── Step 2: Pair images with labels ───────────────────────────────────────────
all_images = sorted(IMG_DIR.glob("*.jpg")) + sorted(IMG_DIR.glob("*.jpeg")) + sorted(IMG_DIR.glob("*.png"))
print(f"[INFO] Found {len(all_images)} images")

paired = []
missing_labels = []

for img in all_images:
    lbl = LBL_DIR / (img.stem + ".txt")
    if lbl.exists():
        paired.append((img, lbl))
    else:
        missing_labels.append(img.name)

print(f"[INFO] {len(paired)} image–label pairs found")

if missing_labels:
    print(f"[WARN] {len(missing_labels)} images have no matching label file (skipped)")

if len(paired) == 0:
    raise SystemExit("[ERROR] No paired files found. Check your folder structure.")

# ── Step 3: Split ─────────────────────────────────────────────────────────────
random.seed(SEED)
random.shuffle(paired)

n       = len(paired)
n_train = int(n * TRAIN_RATIO)
n_val   = int(n * VAL_RATIO)

splits = {
    "train": paired[:n_train],
    "val":   paired[n_train : n_train + n_val],
    "test":  paired[n_train + n_val :],
}

# ── Step 4: Copy files into split folders ─────────────────────────────────────
for split, files in splits.items():
    img_out = Path(OUTPUT_DIR) / split / "images"
    lbl_out = Path(OUTPUT_DIR) / split / "labels"
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    for img_path, lbl_path in files:
        shutil.copy(img_path, img_out / img_path.name)
        shutil.copy(lbl_path, lbl_out / (img_path.stem + ".txt"))

    print(f"  {split:>5}: {len(files)} samples  →  {img_out}")

print("\n[DONE] Dataset split complete!")
print(f"       Output: {OUTPUT_DIR}")

[INFO] Looking for images in : C:\Users\DELL\Desktop\aditya project\MBDD2025/JPEGImages
[INFO] Looking for labels in : C:\Users\DELL\Desktop\aditya project\MBDD2025/Labels


FileNotFoundError: [ERROR] images/ folder not found at: C:\Users\DELL\Desktop\aditya project\MBDD2025/JPEGImages

In [ ]:
"""
MBDD2025 — Script 2: Generate data.yaml for YOLOv8
Run AFTER script 1.
"""

import yaml
from pathlib import Path

# ─── CONFIG ───────────────────────────────────────────────────────────────────
OUTPUT_DIR  = r"C:\Users\DELL\Desktop\aditya project\MBDD2025_split"
YAML_OUTPUT = r"C:\Users\DELL\Desktop\aditya project\mbdd2025.yaml"

# IMPORTANT: These must match the class IDs in your .txt label files.
# Check a few .txt files — first number = class index.
# e.g. "3 0.508 0.354 0.064 0.151" → class 3 = corrosion
CLASS_NAMES = [
    "crack",       # 0
    "leakage",     # 1
    "abscission",  # 2
    "corrosion",   # 3
    "bulge",       # 4
]
# ──────────────────────────────────────────────────────────────────────────────

def generate_yaml():
    # NOTE: YOLOv8 requires forward slashes or raw paths in yaml
    # Using .as_posix() converts Windows backslashes to forward slashes
    split_path = Path(OUTPUT_DIR).as_posix()

    config = {
        "path":  split_path,
        "train": "train/images",
        "val":   "val/images",
        "test":  "test/images",
        "nc":    len(CLASS_NAMES),
        "names": CLASS_NAMES,
    }

    with open(YAML_OUTPUT, "w") as f:
        yaml.dump(config, f, default_flow_style=False, sort_keys=False)

    print(f"[DONE] Saved → {YAML_OUTPUT}")
    print("\nYAML contents:")
    print(yaml.dump(config, default_flow_style=False, sort_keys=False))

    # Quick sanity check — count files per split
    for split in ["train", "val", "test"]:
        imgs = list((Path(OUTPUT_DIR) / split / "images").glob("*.jpg"))
        lbls = list((Path(OUTPUT_DIR) / split / "labels").glob("*.txt"))
        print(f"  {split:>5}: {len(imgs)} images, {len(lbls)} labels")


generate_yaml()

In [ ]:
"""
MBDD2025 — Script 3: Train YOLOv8 Locally
Optimized for NVIDIA GeForce RTX 3050 6GB Laptop GPU.
Run AFTER scripts 1 and 2.

NOTE: 100 epochs on RTX 3050 6GB may take several hours.
      Watch GPU VRAM — if you get CUDA out-of-memory errors,
      reduce BATCH from 8 to 4 below.
"""

import subprocess
subprocess.run(["pip", "install", "ultralytics", "-q"], check=True)

from ultralytics import YOLO
from pathlib import Path
import torch

# ─── CONFIG ───────────────────────────────────────────────────────────────────
YAML_PATH = r"C:\Users\DELL\Desktop\aditya project\mbdd2025.yaml"
MODEL     = "yolov8m.pt"       # auto-downloads on first run
EPOCHS    = 100
IMGSZ     = 640
BATCH     = 8                  # RTX 3050 6GB: 8 is safe; try 4 if OOM error
WORKERS   = 4                  # Windows: keep at 4 or below
PROJECT   = r"C:\Users\DELL\Desktop\aditya project\runs"
RUN_NAME  = "yolov8m_mbdd2025"
# ──────────────────────────────────────────────────────────────────────────────

# ── GPU check ─────────────────────────────────────────────────────────────────
print(f"[INFO] CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"[INFO] GPU            : {torch.cuda.get_device_name(0)}")
    print(f"[INFO] VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    DEVICE = 0
else:
    print("[WARN] No GPU detected — training on CPU (will be very slow!)")
    DEVICE = "cpu"

# ── Auto-resume if last.pt exists, otherwise start fresh ──────────────────────
LAST_PT = Path(PROJECT) / RUN_NAME / "weights" / "last.pt"

if LAST_PT.exists():
    print(f"[INFO] Checkpoint found — resuming training")
    print(f"[INFO] Checkpoint path : {LAST_PT}")
    model = YOLO(str(LAST_PT))
    results = model.train(resume=True)
else:
    print(f"[INFO] No checkpoint found — starting fresh training")
    model = YOLO(MODEL)
    results = model.train(
        data     = YAML_PATH,
        epochs   = EPOCHS,
        imgsz    = IMGSZ,
        batch    = BATCH,
        workers  = WORKERS,
        project  = PROJECT,
        name     = RUN_NAME,
        device   = DEVICE,

        # Augmentation tuned for building defect imagery
        hsv_h    = 0.015,
        hsv_s    = 0.7,
        hsv_v    = 0.4,
        degrees  = 5.0,
        flipud   = 0.1,
        fliplr   = 0.5,
        mosaic   = 1.0,
        mixup    = 0.1,
        auto_augment = "randaugment",

        # Optimizer
        optimizer = "AdamW",
        lr0       = 0.001,
        patience  = 20,
        box       = 9.0,
        save      = True,
        plots     = True,
        verbose   = True,
    )

best_weights = Path(PROJECT) / RUN_NAME / "weights" / "best.pt"
print(f"\n[DONE] Training complete!")
print(f"       Best weights → {best_weights}")
print(f"\n[TIP]  Use best.pt path in scripts 4 and 5:")
print(f"       WEIGHTS = r\"{best_weights}\"")

In [ ]:
"""
MBDD2025 — Script 4: Evaluate on Test Set
Outputs mAP50, mAP50-95, per-class AP, confusion matrix.
Run AFTER script 3.

Update WEIGHTS path below to point to your best.pt file.
"""

import subprocess
subprocess.run(["pip", "install", "ultralytics", "-q"], check=True)

from ultralytics import YOLO
from pathlib import Path
import pandas as pd

# ─── CONFIG ───────────────────────────────────────────────────────────────────
WEIGHTS   = r"C:\Users\DELL\Desktop\aditya project\runs\yolov8m_mbdd2025\weights\best.pt"
YAML_PATH = r"C:\Users\DELL\Desktop\aditya project\mbdd2025.yaml"
RESULTS_DIR = r"C:\Users\DELL\Desktop\aditya project"
IMGSZ     = 640
BATCH     = 8
CLASS_NAMES = ["crack", "leakage", "abscission", "corrosion", "bulge"]
# ──────────────────────────────────────────────────────────────────────────────

# ── Verify weights exist ──────────────────────────────────────────────────────
if not Path(WEIGHTS).exists():
    raise FileNotFoundError(
        f"[ERROR] Weights not found at:\n  {WEIGHTS}\n"
        "  Make sure Script 3 (training) has finished and check the path."
    )

print(f"[INFO] Loading weights from: {WEIGHTS}")
model = YOLO(WEIGHTS)

metrics = model.val(
    data      = YAML_PATH,
    split     = "test",
    imgsz     = IMGSZ,
    batch     = BATCH,
    plots     = True,
    save_json = True,
)

# ── Overall metrics ───────────────────────────────────────────────────────────
print("\n" + "─"*50)
print("  OVERALL TEST SET RESULTS")
print("─"*50)
print(f"  mAP@50       : {metrics.box.map50:.4f}  ({metrics.box.map50*100:.1f}%)")
print(f"  mAP@50-95    : {metrics.box.map:.4f}   ({metrics.box.map*100:.1f}%)")
print(f"  Precision    : {metrics.box.mp:.4f}")
print(f"  Recall       : {metrics.box.mr:.4f}")
print("─"*50)

# ── Per-class table ───────────────────────────────────────────────────────────
rows = []
for name, ap50 in zip(CLASS_NAMES, metrics.box.ap50):
    rows.append({"Defect Class": name, "AP@50": round(ap50, 4), "AP@50 (%)": f"{ap50*100:.1f}%"})

df = pd.DataFrame(rows).sort_values("AP@50", ascending=False)
print("\nPer-Class AP@50:")
print(df.to_string(index=False))

csv_path = Path(RESULTS_DIR) / "per_class_results.csv"
df.to_csv(csv_path, index=False)
print(f"\n[DONE] Per-class results saved → {csv_path}")
print("[INFO] Confusion matrix + curves saved in the runs folder")

In [ ]:
"""
MBDD2025 — Script 5: Inference + Severity Scoring
Runs on test set images and displays results inline in Jupyter.
Run AFTER script 3.

Update WEIGHTS path below to point to your best.pt file.
"""

import subprocess
subprocess.run(["pip", "install", "ultralytics", "-q"], check=True)

import cv2
import random
import numpy as np
import pandas as pd
from pathlib import Path
from ultralytics import YOLO
import matplotlib.pyplot as plt

# ─── CONFIG ───────────────────────────────────────────────────────────────────
WEIGHTS      = r"C:\Users\DELL\Desktop\aditya project\runs\yolov8m_mbdd2025\weights\best.pt"
TEST_IMG_DIR = r"C:\Users\DELL\Desktop\aditya project\MBDD2025_split\test\images"
OUTPUT_DIR   = r"C:\Users\DELL\Desktop\aditya project\inference_results"
RESULTS_DIR  = r"C:\Users\DELL\Desktop\aditya project"
CONF_THRESH  = 0.25
IMGSZ        = 640
DISPLAY_N    = 6       # number of sample images to show inline
CLASS_NAMES  = ["crack", "leakage", "abscission", "corrosion", "bulge"]
# ──────────────────────────────────────────────────────────────────────────────

SEVERITY_COLORS = {
    "HIGH":   "#DC143C",
    "MEDIUM": "#FF8C00",
    "LOW":    "#32CD32",
}
BGR_COLORS = {
    "HIGH":   (0,   20, 220),
    "MEDIUM": (0,  140, 255),
    "LOW":    (50, 205,  50),
}


def iou(boxA, boxB):
    xA = max(boxA[0], boxB[0]); yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2]); yB = min(boxA[3], boxB[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    if inter == 0: return 0.0
    aA = (boxA[2]-boxA[0]) * (boxA[3]-boxA[1])
    aB = (boxB[2]-boxB[0]) * (boxB[3]-boxB[1])
    return inter / (aA + aB - inter)


def compute_severity(x1, y1, x2, y2, img_w, img_h, confidence):
    area_ratio = ((x2 - x1) * (y2 - y1)) / (img_w * img_h)
    if area_ratio > 0.15 or confidence < 0.45:
        return "HIGH", area_ratio
    elif area_ratio >= 0.05:
        return "MEDIUM", area_ratio
    else:
        return "LOW", area_ratio


def annotate_image(image, detections):
    out = image.copy()
    for d in detections:
        x1, y1, x2, y2 = int(d["x1"]), int(d["y1"]), int(d["x2"]), int(d["y2"])
        color = BGR_COLORS[d["severity"]]
        cv2.rectangle(out, (x1, y1), (x2, y2), color, 2)
        label = f"{d['class']} {d['conf']:.2f} | {d['severity']}"
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        cv2.rectangle(out, (x1, y1 - th - 6), (x1 + tw + 4, y1), color, -1)
        cv2.putText(out, label, (x1 + 2, y1 - 4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
    return out


def run_inference():
    # ── Verify paths ──────────────────────────────────────────────────────────
    if not Path(WEIGHTS).exists():
        raise FileNotFoundError(
            f"[ERROR] Weights not found:\n  {WEIGHTS}\n"
            "  Finish Script 3 (training) first."
        )
    if not Path(TEST_IMG_DIR).exists():
        raise FileNotFoundError(
            f"[ERROR] Test images folder not found:\n  {TEST_IMG_DIR}\n"
            "  Run Script 1 (split) first."
        )

    Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
    model = YOLO(WEIGHTS)

    test_images = sorted(Path(TEST_IMG_DIR).glob("*.jpg")) + \
                  sorted(Path(TEST_IMG_DIR).glob("*.jpeg")) + \
                  sorted(Path(TEST_IMG_DIR).glob("*.png"))

    print(f"[INFO] Running inference on {len(test_images)} test images...")

    all_records = []

    for img_path in test_images:
        image = cv2.imread(str(img_path))
        if image is None:
            print(f"[WARN] Could not read: {img_path.name} — skipping")
            continue
        h, w = image.shape[:2]

        # Normal pass
        preds     = model.predict(str(img_path), conf=CONF_THRESH, imgsz=IMGSZ, verbose=False)
        # Lower threshold pass for abscission (class index 2)
        preds_low = model.predict(str(img_path), conf=0.15, imgsz=IMGSZ, verbose=False)

        existing    = [b.xyxy[0].tolist() for b in preds[0].boxes]
        extra_boxes = []
        for box in preds_low[0].boxes:
            if int(box.cls[0]) == 2:
                coords = box.xyxy[0].tolist()
                if not any(iou(coords, eb) > 0.5 for eb in existing):
                    extra_boxes.append(box)

        all_boxes  = list(preds[0].boxes) + extra_boxes
        detections = []

        for box in all_boxes:
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            conf     = float(box.conf[0])
            cls_id   = int(box.cls[0])
            cls_name = CLASS_NAMES[cls_id] if cls_id < len(CLASS_NAMES) else str(cls_id)
            severity, area = compute_severity(x1, y1, x2, y2, w, h, conf)
            detections.append({
                "class": cls_name, "conf": round(conf, 4),
                "severity": severity, "area_pct": round(area * 100, 2),
                "x1": x1, "y1": y1, "x2": x2, "y2": y2,
            })
            all_records.append({
                "image": img_path.name, "class": cls_name,
                "confidence": round(conf, 4), "severity": severity,
                "area_%": round(area * 100, 2),
            })

        annotated = annotate_image(image, detections)
        cv2.imwrite(str(Path(OUTPUT_DIR) / img_path.name), annotated)

    # ── Save CSV report ───────────────────────────────────────────────────────
    df = pd.DataFrame(all_records)
    csv_path = Path(RESULTS_DIR) / "detection_report.csv"
    df.to_csv(csv_path, index=False)

    # ── Print summary ─────────────────────────────────────────────────────────
    print(f"\n{'─'*50}")
    print(f"  Total detections : {len(all_records)}")
    for sev in ["HIGH", "MEDIUM", "LOW"]:
        n = df[df["severity"] == sev].shape[0] if len(df) else 0
        print(f"  {sev:<6} severity   : {n}")
    print(f"\n  Defect breakdown:")
    if len(df):
        print(df["class"].value_counts().to_string())
    print(f"\n  Full report → {csv_path}")
    print(f"  Annotated images → {OUTPUT_DIR}")

    # ── Display sample images inline in Jupyter ───────────────────────────────
    result_imgs = sorted(Path(OUTPUT_DIR).glob("*.jpg"))
    if len(result_imgs) == 0:
        print("[WARN] No output images found to display.")
        return df

    sample = random.sample(result_imgs, min(DISPLAY_N, len(result_imgs)))
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle("MBDD2025 — Sample Detections with Severity Scores", fontsize=14, fontweight="bold")

    for ax, path in zip(axes.flat, sample):
        img = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.set_title(Path(path).name, fontsize=8)
        ax.axis("off")

    for ax in axes.flat[len(sample):]:
        ax.set_visible(False)

    plt.tight_layout()
    grid_path = Path(RESULTS_DIR) / "sample_detections.png"
    plt.savefig(str(grid_path), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"[INFO] Sample grid saved → {grid_path}")

    return df


df_results = run_inference()

# ── Severity distribution chart ────────────────────────────────────────────────
if len(df_results):
    sev_counts = df_results["severity"].value_counts().reindex(["HIGH", "MEDIUM", "LOW"], fill_value=0)
    colors     = [SEVERITY_COLORS[s] for s in sev_counts.index]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.bar(sev_counts.index, sev_counts.values, color=colors, edgecolor="white", linewidth=0.8)
    ax1.set_title("Severity Distribution")
    ax1.set_ylabel("Count")
    for i, v in enumerate(sev_counts.values):
        ax1.text(i, v + 0.5, str(v), ha="center", fontweight="bold")

    class_counts = df_results["class"].value_counts()
    ax2.barh(class_counts.index, class_counts.values, color="#4C72B0")
    ax2.set_title("Defect Type Distribution")
    ax2.set_xlabel("Count")

    plt.tight_layout()
    dist_path = Path(RESULTS_DIR) / "severity_distribution.png"
    plt.savefig(str(dist_path), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"[INFO] Distribution chart saved → {dist_path}")